In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    brier_score_loss
)
from lightgbm import LGBMClassifier


# ============================================================
# Config
# ============================================================
INPUT_PATH = Path("./training_data_standardization_with_pairwise_products.csv")
LABEL_COL = "label"

SEEDS = [35, 42, 55, 64, 100]

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2

ECE_BINS = 15


# ============================================================
# Utils
# ============================================================
def remove_index_like_columns(df):
    df = df.copy()
    drop_cols = []

    for col in df.columns:
        col_lower = str(col).lower()

        if col_lower.startswith("unnamed") or col_lower in ["index", "level_0"]:
            drop_cols.append(col)
            continue

        values = df[col].values
        if pd.api.types.is_numeric_dtype(df[col]):
            seq0 = np.arange(len(df))
            seq1 = np.arange(1, len(df) + 1)
            if np.array_equal(values, seq0) or np.array_equal(values, seq1):
                drop_cols.append(col)

    if drop_cols:
        df = df.drop(columns=drop_cols)

    return df


def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    n = len(y_true)

    for b in range(n_bins):
        mask = bin_ids == b
        if np.sum(mask) == 0:
            continue

        acc = np.mean(y_true[mask])
        conf = np.mean(y_prob[mask])
        weight = np.sum(mask) / n

        ece += abs(acc - conf) * weight

    return float(ece)


def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 99)

    best_th = 0.5
    best_f1 = -1

    for th in thresholds:
        pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = th

    return best_th


def evaluate(y_true, y_prob, threshold):
    pred = (y_prob >= threshold).astype(int)

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Brier": brier_score_loss(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob)
    }


# ============================================================
# Load
# ============================================================
df = pd.read_csv(INPUT_PATH)
df = remove_index_like_columns(df)

df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

feature_cols = [c for c in df.columns if c != LABEL_COL]
X = df[feature_cols].copy()
y = df[LABEL_COL].copy()

# 결측 처리
X = X.fillna(X.median(numeric_only=True))

# 비수치형 처리
for col in X.select_dtypes(exclude=[np.number]).columns:
    X[col] = pd.factorize(X[col])[0]

X = X.astype(float)

# ============================================================
# Split (고정)
# ============================================================
n = len(df)

train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]


# ============================================================
# Run per seed
# ============================================================
results = []

for seed in SEEDS:

    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    scale_pos_weight = neg / pos if pos > 0 else 1.0

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    # threshold (val 기준)
    val_prob = model.predict_proba(X_val)[:, 1]
    best_th = find_best_threshold(y_val.values, val_prob)

    # test 평가
    test_prob = model.predict_proba(X_test)[:, 1]
    metrics = evaluate(y_test.values, test_prob, best_th)

    row = {
        "seed": seed,
        "threshold": best_th,
        **metrics
    }

    results.append(row)

    print(f"[SEED {seed}] done")


# ============================================================
# 결과 정리 + mean row 추가
# ============================================================
df_results = pd.DataFrame(results)

mean_row = df_results.mean(numeric_only=True)
mean_row["seed"] = "mean"

df_results = pd.concat(
    [df_results, pd.DataFrame([mean_row])],
    ignore_index=True
)

# ============================================================
# Save
# ============================================================
OUT_PATH = Path("./lgbm_pairwise_product_metrics_seeds.csv")
df_results.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\n=== Final Result ===")
print(df_results)

print(f"\n저장 완료: {OUT_PATH.resolve()}")

[SEED 35] done
[SEED 42] done
[SEED 55] done
[SEED 64] done
[SEED 100] done

=== Final Result ===
   seed  threshold     AUROC     AUPRC        F1     Brier       ECE
0    35      0.780  0.831945  0.336940  0.381395  0.037321  0.037605
1    42      0.810  0.825470  0.339490  0.357488  0.036872  0.036959
2    55      0.820  0.834300  0.340984  0.380488  0.037846  0.037691
3    64      0.840  0.828787  0.339883  0.362694  0.036978  0.036897
4   100      0.880  0.838647  0.345370  0.346939  0.037533  0.038376
5  mean      0.826  0.831830  0.340533  0.365801  0.037310  0.037506

저장 완료: D:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\14_no_weight_experiments\lgbm_pairwise_product_metrics_seeds.csv
